## Rolling Updates

In this notebook, we look at Rolling Updates in Kubernetes.

- Make sure you have a Kubernetes cluster (Docker Desktop) running.
- Also make sure you have installed the `kubectl` tool on your computer.

## Build custom Docker images on the host machine

- Builds a Docker image `hello-app:1.0` from the subfolder `v1`
  - The Dockerfile uses `nginx:latest` as the base image, and sets the default web page to `Hello (version 1)`.
- Builds a Docker image `hello-app:2.0` from the subfolder `v2`
  - The Dockerfile uses `nginx:latest` as the base image, and sets the default web page to `Hello (version 2)`.

In [1]:
!docker build -t hello-app:1.0 -f v1/Dockerfile v1/.
!docker build -t hello-app:2.0 -f v2/Dockerfile v2/.
!docker image ls | grep hello-app

#0 building with "desktop-linux" instance using docker driver

#1 [internal] load build definition from Dockerfile
#1 transferring dockerfile: 113B 0.0s done
#1 DONE 0.1s

#2 [internal] load metadata for docker.io/library/nginx:latest
#2 DONE 0.0s

#3 [internal] load .dockerignore
#3 transferring context: 2B done
#3 DONE 0.0s

#4 [internal] load build context
#4 transferring context: 54B 0.0s done
#4 DONE 0.1s

#5 [1/3] FROM docker.io/library/nginx:latest
#5 DONE 0.1s

#6 [2/3] WORKDIR /usr/share/nginx/html
#6 DONE 0.0s

#7 [3/3] COPY index.html index.html
#7 DONE 0.1s

#8 exporting to image
#8 exporting layers 0.1s done
#8 writing image sha256:80a266492983016feff607cb5ff751f7e11981674005d84033fb4850ce7388ff done
#8 naming to docker.io/library/hello-app:1.0 done
#8 DONE 0.1s

View build details: docker-desktop://dashboard/build/desktop-linux/desktop-linux/bwsnnjmk37w341r0juxbc0j9e
#0 building with "desktop-linux" instance using docker driver

#1 [internal] load build definition from Do

## Create a Deployment (version 1)

- The Deployment definition is in the YAML file `manifests/hello-deployment.yaml`.

In [2]:
!kubectl create -f manifests/hello-deployment.yaml

deployment.apps/hello-dep created


## Let's look at the Deployment's YAML.

**Note:**

- The number of `replicas` is set to 3.
- The Pod template's `labels` are `app: hello-dep`
  - The Deployment's `matchLabels` match these labels
    - Therefore, the Deployment will create 3 replicas from the Pod template.
  - The Deployment uses a `RollingUpdate` `strategy` `type` with `maxSurge` and `maxUnavailable` set to 1.
    - Therefore, there can exist:
      - A maximum of 4 (`replicas` + `maxSurege`) Pod instances during the update.
      - A minimum of 2 (`replicas` - `maxUnavailable`) Pod instances during the update.
- The Deployment uses a `revisionHistoryLimit` of 3.
  - Therefore, a maximum of 3 ReplicaSet versions will be kept in the revistion history.
- The Pod template's containers are:
  - Based on the Docker `hello-app:1.0` image (version 1).
  - Listening on `containerPort` 8080.

```bash
apiVersion: apps/v1
kind: Deployment
metadata:
  name: hello-dep                # the Deployment's name
  namespace: default
spec:
  replicas: 3                    # the Deployment defines three Pod replicas
  revisionHistoryLimit: 3        # the Deployment defines a maximum number of ReplicaSet revisions in history to 3
  strategy:
    type: RollingUpdate          # the Deployment uses a RollingUpdate strategy
    rollingUpdate:
      maxSurge: 1                # a maximum of 3+1 Pod instances can exist during the update
      maxUnavailable: 1          # a minimum of 3-1 Pod instances can exist during the update
  selector:
    matchLabels:
      app: hello-dep             # the Deployment's matchLabels match the Pod template's labels below
  template:
    metadata:
      labels:
        app: hello-dep           # the Pod template defines one label (app: hello-dep)
    spec:
      containers:
      - image: hello-app:1.0     # the Pod template's containers are based on the hello-app:1.0 image (version 1)
        resources:
          requests:
            cpu: 100m
            memory: 128Mi
          limits:
            cpu: 250m
            memory: 256Mi      
        imagePullPolicy: Never
        name: hello-dep          # the container's name
        ports:
        - containerPort: 8080    # the containers are listening on port 8080
```

In [3]:
!type manifests\hello-deployment.yaml
#!cat manifests/hello-deployment.yaml # use this on Linux/Mac

apiVersion: apps/v1
kind: Deployment
metadata:
  name: hello-dep
  namespace: default
spec:
  replicas: 3
  revisionHistoryLimit: 3
  strategy:
    type: RollingUpdate
    rollingUpdate:
      maxSurge: 1
      maxUnavailable: 1
  selector:
    matchLabels:
      app: hello-dep
  template:
    metadata:
      labels:
        app: hello-dep
    spec:
      containers:
      - image: hello-app:1.0 # change this to: hello-app:2.0
        resources:
          requests:
            cpu: 100m
            memory: 128Mi
          limits:
            cpu: 250m
            memory: 256Mi      
        imagePullPolicy: Never
        name: hello-dep
        ports:
        - containerPort: 8080


## Get Deployment rollout status

- Note that the Deployment was rolled out successfully.

In [4]:
!kubectl rollout status deployment/hello-dep

deployment "hello-dep" successfully rolled out


## List Pods

- We see that 3 Pod replicas are running.

In [5]:
#!kubectl get po -o wide
!kubectl get pods -o wide

NAME                         READY   STATUS    RESTARTS   AGE   IP          NODE             NOMINATED NODE   READINESS GATES
hello-dep-869f9cb557-m447k   1/1     Running   0          51s   10.1.1.11   docker-desktop   <none>           <none>
hello-dep-869f9cb557-qpfb4   1/1     Running   0          51s   10.1.1.9    docker-desktop   <none>           <none>
hello-dep-869f9cb557-v7ks7   1/1     Running   0          51s   10.1.1.10   docker-desktop   <none>           <none>


## Get the default web page in one of the pods

- Use one of the pod names (above) to execute the command `-- curl http://localhost` in its container.
- We see that the default web page returns `Hello (version 1)` as defined in image `hello-app:1.0`.

In [9]:
POD_NAME = !kubectl get pods -o jsonpath="{.items[0].metadata.name}"
POD_NAME = POD_NAME[0]
!kubectl exec {POD_NAME} -- curl http://localhost

Hello (version 1)


  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed

  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
100    17  100    17    0     0  42288      0 --:--:-- --:--:-- --:--:-- 17000


## List ReplicaSets

- We see that the ReplicaSet contains 3 Pod replicas with containers based on image `hello-app:1.0`.

In [10]:
#!kubectl get rs -o wide
!kubectl get replicasets -o wide

NAME                   DESIRED   CURRENT   READY   AGE     CONTAINERS   IMAGES          SELECTOR
hello-dep-869f9cb557   3         3         3       4m12s   hello-dep    hello-app:1.0   app=hello-dep,pod-template-hash=869f9cb557


## Create a Deployment (version 2)

- Open the YAML file `manifests/hello-deployment.yaml`.
- Change the image version from `1.0` to `2.0`.
- Save the file.

```bash
apiVersion: apps/v1
kind: Deployment
metadata:
  name: hello-dep
  namespace: default
spec:
  replicas: 3
  revisionHistoryLimit: 3
  strategy:
    type: RollingUpdate
    rollingUpdate:
      maxSurge: 1
      maxUnavailable: 1
  selector:
    matchLabels:
      app: hello-dep
  template:
    metadata:
      labels:
        app: hello-dep
    spec:
      containers:
      - image: hello-app:2.0 # change this to: hello-app:2.0
        resources:
          requests:
            cpu: 100m
            memory: 128Mi
          limits:
            cpu: 250m
            memory: 256Mi      
        imagePullPolicy: Never
        name: hello-dep
        ports:
        - containerPort: 8080
```

In [11]:
!type manifests\hello-deployment.yaml
#!cat manifests/hello-deployment.yaml # use this on Linux/Mac

apiVersion: apps/v1
kind: Deployment
metadata:
  name: hello-dep
  namespace: default
spec:
  replicas: 3
  revisionHistoryLimit: 3
  strategy:
    type: RollingUpdate
    rollingUpdate:
      maxSurge: 1
      maxUnavailable: 1
  selector:
    matchLabels:
      app: hello-dep
  template:
    metadata:
      labels:
        app: hello-dep
    spec:
      containers:
      - image: hello-app:2.0 # change this to: hello-app:2.0
        resources:
          requests:
            cpu: 100m
            memory: 128Mi
          limits:
            cpu: 250m
            memory: 256Mi      
        imagePullPolicy: Never
        name: hello-dep
        ports:
        - containerPort: 8080


## Deploy the updated Deployment's YAML file

In [12]:
!kubectl apply -f manifests/hello-deployment.yaml

deployment.apps/hello-dep configured

## Get Deployment rollout status

- We see that the updated Deployment's YAML definition was rolled out successfully.

In [13]:
!kubectl rollout status deployment/hello-dep

deployment "hello-dep" successfully rolled out


## List Pods

- We see that three Pods are running.

In [14]:
#!kubectl get po -o wide
!kubectl get pods -o wide

NAME                         READY   STATUS    RESTARTS   AGE   IP          NODE             NOMINATED NODE   READINESS GATES
hello-dep-5dc78f9b6b-hf2ch   1/1     Running   0          12s   10.1.1.14   docker-desktop   <none>           <none>
hello-dep-5dc78f9b6b-hsxm5   1/1     Running   0          13s   10.1.1.12   docker-desktop   <none>           <none>
hello-dep-5dc78f9b6b-v7hxd   1/1     Running   0          13s   10.1.1.13   docker-desktop   <none>           <none>


## Get the default web page in one of the pods

- Use one of the pod names (above) to execute the command `-- curl http://localhost` in its container.
- We see that the default web page returns `Hello (version 2)` as defined in image `hello-app:2.0`.

In [15]:
POD_NAME = !kubectl get pods -o jsonpath="{.items[0].metadata.name}"
POD_NAME = POD_NAME[0]
!kubectl exec {POD_NAME} -- curl http://localhost

Hello (version 2)


  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed

  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
100    17  100    17    0     0  43701      0 --:--:-- --:--:-- --:--:-- 17000


## List ReplicaSets

- Notice that:
  - The current ReplicaSet contains 3 Pod replicas with containers based on image `hello-app:2.0`.
  - The previous ReplicaSet is kept in the revision history with `DESIRED`, `CURRENT` and `READY` set to 0 (no Pods running).
    - This is because `revisionHistoryLimit` is set to 3 in the Deployment's YAML file.

In [16]:
#!kubectl get rs -o wide
!kubectl get replicasets -o wide

NAME                   DESIRED   CURRENT   READY   AGE     CONTAINERS   IMAGES          SELECTOR
hello-dep-5dc78f9b6b   3         3         3       34s     hello-dep    hello-app:2.0   app=hello-dep,pod-template-hash=5dc78f9b6b
hello-dep-869f9cb557   0         0         0       6m12s   hello-dep    hello-app:1.0   app=hello-dep,pod-template-hash=869f9cb557


## Rollback the Deployment

- Here we are using `--to-revision 1` to roll back the Deployment to version 1.

In [17]:
#!kubectl rollout undo deployment/hello-dep
!kubectl rollout undo deployment/hello-dep --to-revision 1

deployment.apps/hello-dep rolled back


## Get Deployment rollout status

- We see that the rolled back Deployment was rolled out successfully.

In [18]:
!kubectl rollout status deployment/hello-dep

deployment "hello-dep" successfully rolled out


## List Pods

- We see that 3 Pods are running.

In [19]:
#!kubectl get po -o wide
!kubectl get pods -o wide

NAME                         READY   STATUS    RESTARTS   AGE   IP          NODE             NOMINATED NODE   READINESS GATES
hello-dep-869f9cb557-bkcnx   1/1     Running   0          14s   10.1.1.16   docker-desktop   <none>           <none>
hello-dep-869f9cb557-gvwp9   1/1     Running   0          13s   10.1.1.17   docker-desktop   <none>           <none>
hello-dep-869f9cb557-l66nv   1/1     Running   0          14s   10.1.1.15   docker-desktop   <none>           <none>


## Get the default web page in one of the pods

- Use one of the pod names (above) to execute the command `-- curl http://localhost` in its container.
- We see that the default web page returns `Hello (version 1)` as defined in image `hello-app:1.0`.

In [20]:
POD_NAME = !kubectl get pods -o jsonpath="{.items[0].metadata.name}"
POD_NAME = POD_NAME[0]
!kubectl exec {POD_NAME} -- curl http://localhost

Hello (version 1)


  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed

  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
100    17  100    17    0     0  35343      0 --:--:-- --:--:-- --:--:-- 17000


## List ReplicaSets

- We see that the ReplicaSet contains 3 Pod replicas with containers based on image `hello-app:1.0`.

In [21]:
#!kubectl get rs -o wide
!kubectl get replicasets -o wide

NAME                   DESIRED   CURRENT   READY   AGE     CONTAINERS   IMAGES          SELECTOR
hello-dep-5dc78f9b6b   0         0         0       92s     hello-dep    hello-app:2.0   app=hello-dep,pod-template-hash=5dc78f9b6b
hello-dep-869f9cb557   3         3         3       7m10s   hello-dep    hello-app:1.0   app=hello-dep,pod-template-hash=869f9cb557


## Delete the Deployment

In [22]:
!kubectl delete -f manifests/hello-deployment.yaml

deployment.apps "hello-dep" deleted


## List Deployments, ReplicaSets and Pods

- We see that deleting the Deployment also deleted its associated ReplicaSet and Pods.

In [23]:
#!kubectl get deploy
#!kubectl get rs
#!kubectl get po
!kubectl get deployments
!kubectl get replicasets
!kubectl get pods

No resources found in default namespace.
No resources found in default namespace.
No resources found in default namespace.


## Delete the custom Docker images from the host machine

In [ ]:
!docker rmi hello-app:1.0
!docker rmi hello-app:2.0
!docker image ls

Untagged: hello-app:1.0
Deleted: sha256:80a266492983016feff607cb5ff751f7e11981674005d84033fb4850ce7388ff
Untagged: hello-app:2.0
Deleted: sha256:e3de9e1cad51e264f74f8267db810e5a8e0159f5bc56f01295755a8814259f01
REPOSITORY   TAG       IMAGE ID   CREATED   SIZE
